# GradientQL: scan DVGA in Colab

This notebook runs [**GradientQL**](https://github.com/ccc909/GradientQL), an autonomous GraphQL
vulnerability scanner driven by a single language model, against a local copy of the
[Damn Vulnerable GraphQL Application (DVGA)](https://github.com/dolevf/Damn-Vulnerable-GraphQL-Application):
an intentionally vulnerable target that is safe to attack.

You need an [OpenRouter](https://openrouter.ai) API key. The scan makes model calls that cost money
(a short run is a few cents). Run the cells top to bottom.

> ⚠️ **Only ever point GradientQL at a target you own or are authorized to test.** This notebook
> scans the bundled DVGA on localhost and nothing else.


In [ ]:
#@title 1. Install GradientQL
!pip install -q "git+https://github.com/ccc909/GradientQL.git"
print("GradientQL installed.")


In [ ]:
#@title 2. Start DVGA (the target) in the background  [first run takes a few minutes]
import pathlib
import subprocess
import time
import urllib.request

URL = "http://127.0.0.1:5013/graphql"
LOG = "/content/dvga_server.log"

# Run the prebuilt DVGA image via udocker (userspace, no Docker daemon), so it brings its own Python.
subprocess.run(["pip", "install", "-q", "udocker"], check=True)
subprocess.run(["udocker", "--allow-root", "install"], check=True)
print("[1/2] preparing the DVGA image (first run pulls a few hundred MB) ...", flush=True)
subprocess.run(["udocker", "--allow-root", "pull", "dolevf/dvga"], check=True)
subprocess.run(["udocker", "--allow-root", "create", "--name=dvga", "dolevf/dvga"],
               capture_output=True)  # harmless if it already exists
subprocess.run(["udocker", "--allow-root", "setup", "--execmode=P2", "dvga"], check=True)  # P2: Colab

# Patch DVGA (idempotent) so one blocking request can't wedge the server, then launch it.
print("[2/2] starting DVGA ...", flush=True)
launch = ("grep -q monkey.patch_all /opt/dvga/app.py || "
          "sed -i '1ifrom gevent import monkey; monkey.patch_all()' /opt/dvga/app.py; "
          "cd /opt/dvga && WEB_HOST=127.0.0.1 WEB_PORT=5013 python app.py")
logf = open(LOG, "wb")
subprocess.Popen(["udocker", "--allow-root", "run", "dvga", "sh", "-c", launch],
                 stdout=logf, stderr=subprocess.STDOUT)

probe = urllib.request.Request(URL, data=b'{"query":"{__typename}"}',
                               headers={"Content-Type": "application/json"})
for _ in range(90):
    try:
        if urllib.request.urlopen(probe, timeout=3).status == 200:
            print("DVGA is up at", URL)
            break
    except Exception:
        time.sleep(2)
else:
    print("DVGA did not come up. Last lines of its server log:\n")
    print(pathlib.Path(LOG).read_text()[-3000:])
    raise RuntimeError("DVGA did not start (see the log above).")


In [ ]:
#@title 3. Enter your OpenRouter API key
import getpass
import os
os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key (input hidden): ").strip()
print("Key set." if os.environ["OPENROUTER_API_KEY"] else "No key entered - the scan will not run.")


In [ ]:
#@title 4. Run the scan
MODEL = "z-ai/glm-5.2"  #@param ["z-ai/glm-5.2", "openai/gpt-oss-120b", "qwen/qwen3.7-max"]
BUDGET = 20  #@param {type:"slider", min:5, max:60, step:5}

with open("demo.yaml", "w") as f:
    f.write(f"scanner:\n  budget: {BUDGET}\nllm:\n  attacker_model: \"{MODEL}\"\n")

# --no-tui: plain logs (no terminal in Colab); --trace: write the step log.
!python -m gradientql --settings demo.yaml --url http://127.0.0.1:5013/graphql --no-tui --trace


## What just happened

GradientQL introspected DVGA, worked through the attack surface on its own, and printed a report of
what it confirmed and retracted. Because `--trace` was on, a full step-by-step log was written under
`output/` (open the file browser on the left of Colab); the findings alone are in
`output/vuln_stream.jsonl`.

- **Go deeper:** raise `BUDGET` and re-run the scan cell. glm finds the most; `gpt-oss-120b` is the
  cheapest and fastest.
- **Scan your own target:** replace the `--url` with an endpoint you are authorized to test, and skip
  the DVGA cell.
- **More options:** see the [README](https://github.com/ccc909/GradientQL) and
  [Configuration](https://github.com/ccc909/GradientQL/blob/main/docs/CONFIGURATION.md).
